In [4]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field
from typing import Literal
load_dotenv()

True

In [ ]:
model = ChatOpenAI(model="gpt-4o-mini")
# model.invoke('hi')

AIMessage(content='Hello! How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 8, 'total_tokens': 17, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_20370cf6b8', 'id': 'chatcmpl-EIDcroKHNV10NFpfGa5VG8ZZqz2dY', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a04dc2-85ac-7641-aa44-c7f2a2677788-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 8, 'output_tokens': 9, 'total_tokens': 17, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [76]:
# custom output format 
class COTResult(BaseModel):
    category: Literal['Billing','Technical','Feature Request','Churn Risk','Spam','Other']
    urgency: Literal['High', 'Medium', 'Low']
    confidence: int = Field(ge=1, le=10)
    reasoning: str = Field(description='One sentence explanation')
    step_thinking: str = Field(description='Step-by-step reasoning process')

parser = PydanticOutputParser(pydantic_object=COTResult)

In [93]:
print(parser.get_format_instructions())

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"properties": {"category": {"enum": ["Billing", "Technical", "Feature Request", "Churn Risk", "Spam", "Other"], "title": "Category", "type": "string"}, "urgency": {"enum": ["High", "Medium", "Low"], "title": "Urgency", "type": "string"}, "confidence": {"maximum": 10, "minimum": 1, "title": "Confidence", "type": "integer"}, "reasoning": {"description": "One sentence explanation", "title": "Reasoning", "type": "string"}, "step_thinking": {"description": "Step-by-step reasoning process", "title": "Step Thinking", "type": "string"}}, "required": 

In [ ]:
COT_PROMPT= ChatPromptTemplate.from_messages([
  ('system','''
 Your are a expert support email classifier.
   RULES:
   - Login broken after payment → Billing (NOT Technical)
   - App crashes → Technical
   - Pricing + evaluating alternatives → Churn Risk
   - Feature requests → Feature Request
   - Spam → Spam

 {format_instructions}  ,
   ('Human', 'Subject: {subject}\nBody: {body}')

''')

]).partial(format_instructions = parser.get_format_instructions())

In [78]:
chain = COT_PROMPT | model | parser

In [79]:
# result = chain.invoke({
#     "subject": "App crashes",
#     "body": "I'm experiencing crashes when I try to open the app."
# })

In [80]:
import pandas as pd
data = pd.read_excel("emails.xlsx")
data.head(10)

,ID,Subject,Sender,Body,Urgency,Category
0,1,Invoice Discrepancy,finance@company.com,"My invoice is wrong, please fix it.",High,NaN
1,2,Need to Update Payment Method,billing@client.com,I need to update my payment method for the sub...,Medium,NaN
2,3,Subscription Cancellation,support@business.com,Please cancel my subscription immediately.,High,NaN
3,4,Invoice Not Received,admin@company.com,I haven't received my latest invoice. Can you ...,Medium,NaN
4,5,App Crashes on Save,dev@client.com,The app crashes when I click Save. Please assist.,High,NaN
5,6,Login Issues,user@company.com,I can't log into my account. It keeps showing ...,High,NaN
6,7,Feature Not Working,team@business.com,The report generation feature is not working a...,Medium,NaN
7,8,App Freezing,user@client.com,The app keeps freezing when I try to open it.,High,NaN
8,9,Request for Dark Mode,design@company.com,Please add dark mode to the application.,Medium,NaN
9,10,Add Custom Fields,pm@business.com,Can we add custom fields to the project manage...,Medium,NaN


In [ ]:
# lst1 = [2,3,6]
# lst2 = [4,5,6]

# for i , j in zip(lst1, lst2):
#     print(f"Element from lst1: {i}, Element from lst2: {j}")

Element from lst1: 2, Element from lst2: 4
Element from lst1: 3, Element from lst2: 5
Element from lst1: 6, Element from lst2: 6


In [89]:
subject = data['Subject'].tolist()
body = data['Body'].tolist()

# subject

In [ ]:
# subject = data['Subject'].tolist()
# body = data['Body'].tolist()
# for i, j in zip(subject, body):
#     print(f"Subject: {i}\nBody: {j}\n")
# email_data
# print(COT_PROMPT.messages[0].prompt.template)
# email_data[:10]

In [81]:
subject = data['Subject'].tolist()
body = data['Body'].tolist()
# dictionary
email_data = [{"subject": sub, "body": bod} for sub, bod in zip(subject, body)]

In [94]:
result = chain.batch(email_data,
                     config={"batch_size": 10})

In [ ]:
# result

[COTResult(category='Billing', urgency='High', confidence=9, reasoning="The user's concern about an invoice being wrong relates directly to billing issues.", step_thinking='1. Identify the issue in the email (invoice discrepancy). 2. Determine if it pertains to billing or technical support. 3. Classify as Billing due to its nature.'),
 COTResult(category='Billing', urgency='Medium', confidence=8, reasoning='The user wants to update their payment method, which pertains to billing.', step_thinking="1. Identify the user's request about payment method.\n2. Classify the issue as billing-related since it involves payment updates."),
 COTResult(category='Churn Risk', urgency='High', confidence=9, reasoning='The user is requesting to cancel their subscription, indicating potential dissatisfaction.', step_thinking='1. The user explicitly states they want to cancel their subscription. 2. Cancellation requests often indicate a risk of churn. 3. Therefore, classify this as Churn Risk.'),
 COTResul

In [83]:
# write it to the table or in files
category = []
urgency = []
confidence = []
reasoning = []
for i in result:
    category.append(i.category)
    urgency.append(i.urgency)
    confidence.append(i.confidence)
    reasoning.append(i.reasoning)


In [84]:
data['Category'] = category
data['Urgency'] = urgency
data['Confidence'] = confidence
data['Reasoning'] = reasoning

In [86]:
data.to_csv("output.csv", index=False)

In [107]:
# 
print(model.invoke('I received a billing complaint from John about invoice #1042. and please remeber invoce number for future').content)

I can't store information like invoice numbers for future reference, but I can help you address the billing complaint regarding invoice #1042. What specific issues did John raise about the invoice?


In [105]:
print(model.invoke("what is his invoice number.").content)

I'm sorry, but I can't provide specific invoice numbers or confidential information. If you have a particular context in mind or need help with something else related to invoices, feel free to share!


In [106]:
model.invoke(
    "I received a billing complaint from John about invoice #1042. "
    "what is his invoice number."
).content

"John's invoice number is #1042."

In [109]:
user_turn_1 = "I received a billing complaint from John about invoice #1042. He was charged $500 instead of $250."
ai_turn_1 = model.invoke(user_turn_1).content
ai_turn_1


'To address John’s billing complaint regarding invoice #1042, follow these steps:\n\n1. **Verify the Invoice**: Check the details of invoice #1042 to confirm the amounts charged. Ensure that the original charge was indeed $500 and verify the reason for the discrepancy.\n\n2. **Review Charge Details**: Look into any applicable discounts, promotions, or pricing agreements that could explain the $250 charge he expected.\n\n3. **Contact John**: Reach out to John to acknowledge his complaint. Explain that you are looking into the issue and provide a timeline for when he can expect an update.\n\n4. **Adjust the Invoice if Necessary**: If you find that John was incorrectly charged, adjust the invoice to reflect the correct amount of $250. Prepare to issue a refund or credit for the difference if needed.\n\n5. **Communicate the Resolution**: Once you have identified the issue, communicate the resolution to John. If the charge was an error, provide a clear explanation and confirmation of the ad

In [ ]:
user_turn_2 = "What category does his complaint fall under?"
# ai_turn_2 = model.invoke(user_turn_2).content
# ai_turn_2

In [115]:
prompt = f"""
Previous conversation:
User: {user_turn_1}
AI: {ai_turn_1}

Current user query: {user_turn_2}
"""

# print(prompt)

ai_turn_2 = model.invoke(prompt).content
ai_turn_2

"John's complaint regarding invoice #1042 falls under the category of **billing disputes** or **customer billing issues**. This category typically includes complaints about incorrect charges, discrepancies in invoices, unauthorized transactions, or issues related to payment processing. Addressing such complaints is crucial for maintaining customer satisfaction and trust."

In [116]:
user_turn_3 = 'summarize the tickets'

In [117]:
prompt = f"""
Previous conversation:
User: {user_turn_1}
AI: {ai_turn_1}
User: {user_turn_2}
AI: {ai_turn_2}

Current user query: {user_turn_3}
"""

print(prompt)


Previous conversation:
User: I received a billing complaint from John about invoice #1042. He was charged $500 instead of $250.
AI: To address John’s billing complaint regarding invoice #1042, follow these steps:

1. **Verify the Invoice**: Check the details of invoice #1042 to confirm the amounts charged. Ensure that the original charge was indeed $500 and verify the reason for the discrepancy.

2. **Review Charge Details**: Look into any applicable discounts, promotions, or pricing agreements that could explain the $250 charge he expected.

3. **Contact John**: Reach out to John to acknowledge his complaint. Explain that you are looking into the issue and provide a timeline for when he can expect an update.

4. **Adjust the Invoice if Necessary**: If you find that John was incorrectly charged, adjust the invoice to reflect the correct amount of $250. Prepare to issue a refund or credit for the difference if needed.

5. **Communicate the Resolution**: Once you have identified the iss

In [118]:
model.invoke(prompt).content

'To summarize the tickets related to John’s complaint about invoice #1042:\n\n1. **Ticket ID**: Complaint about Invoice #1042\n   - **Customer**: John\n   - **Issue**: Incorrect charge of $500 instead of the expected $250\n   - **Category**: Billing disputes / Customer billing issues\n   - **Resolution Steps**:\n     - Verified invoice details to confirm the charged amount\n     - Reviewed applicable discounts and pricing agreements\n     - Communicated with John regarding the complaint\n     - Adjusted invoice if necessary\n     - Issued refund or credit if an error was identified\n     - Implemented preventative measures to avoid future issues\n     - Followed up with John to ensure satisfaction after resolution\n\nThis summarizes the complaint and the intended response process.'

In [12]:
class TOTResult(BaseModel):
    email: str
    subject: str
    body: str


parser_2 = PydanticOutputParser(pydantic_object=TOTResult)
print(parser_2.get_format_instructions())

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"properties": {"email": {"title": "Email", "type": "string"}, "subject": {"title": "Subject", "type": "string"}, "body": {"title": "Body", "type": "string"}}, "required": ["email", "subject", "body"]}
```


In [ ]:
# print(parser.get_format_instructions())

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"properties": {"category": {"enum": ["Billing", "Technical", "Feature Request", "Churn Risk", "Spam", "Other"], "title": "Category", "type": "string"}, "urgency": {"enum": ["High", "Medium", "Low"], "title": "Urgency", "type": "string"}, "confidence": {"maximum": 10, "minimum": 1, "title": "Confidence", "type": "integer"}, "reasoning": {"description": "One sentence explanation", "title": "Reasoning", "type": "string"}, "step_thinking": {"description": "Step-by-step reasoning process", "title": "Step Thinking", "type": "string"}}, "required": 